# Bài 23 — Application: Transformer text classification

Huấn luyện một Transformer Encoder tối giản để phân loại cảm xúc review IMDb thành `negative` hoặc `positive`. Luồng xử lý: raw text → tokenizer → token IDs → embedding + positional encoding → Transformer Encoder → mean pooling → linear classifier.

Notebook đánh giá trên test set và thử dự đoán ít nhất năm trường hợp: câu ngắn, câu dài, phủ định, tương phản và cảm xúc mơ hồ. Kết quả các câu ví dụ là dự đoán của mô hình sau khi bạn chạy notebook; không giả định mô hình luôn đúng.


In [ ]:
import os, random, tarfile, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


TensorFlow: 2.21.0
GPU: []


## 1. Tải và chuẩn bị IMDb

Notebook thử tải ACL IMDb từ Stanford. Nếu kết nối bị từ chối, sẽ chuyển sang IMDb của Keras/TensorFlow. Validation được tách stratified từ train; test chỉ dùng để đánh giá cuối.


In [ ]:
DATA_DIR = Path('/content/aclImdb')
ARCHIVE = Path('/content/aclImdb_v1.tar.gz')
DATA_URL = 'https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'

if not (DATA_DIR / 'train' / 'pos').exists():
    try:
        if not ARCHIVE.exists():
            print('Đang tải ACL IMDb từ Stanford (~84 MB)...')
            urllib.request.urlretrieve(DATA_URL, ARCHIVE)
        with tarfile.open(ARCHIVE, 'r:gz') as archive:
            archive.extractall('/content', filter='data')
    except Exception as err:
        ARCHIVE.unlink(missing_ok=True)
        print('Không tải được Stanford:', repr(err))
        print('Đang thử bản IMDb qua Keras/TensorFlow...')
        try:
            (x_train_raw, y_train_raw), (x_test_raw, y_test) = tf.keras.datasets.imdb.load_data(seed=SEED)
            word_index = tf.keras.datasets.imdb.get_word_index()
            reverse_index = {i + 3: w for w, i in word_index.items()}
            def decode_review(seq):
                return ' '.join(reverse_index.get(int(i), '<UNK>') for i in seq if int(i) > 3)
            all_train_texts = [decode_review(s) for s in x_train_raw]
            all_train_labels = np.asarray(y_train_raw, dtype=np.int32)
            test_texts = [decode_review(s) for s in x_test_raw]
            y_test = np.asarray(y_test, dtype=np.int32)
        except Exception as fallback_err:
            raise RuntimeError('Không tải được IMDb từ Stanford hoặc Keras. Kiểm tra kết nối Internet của Colab rồi chạy lại.') from fallback_err

if 'all_train_texts' not in globals():
    def read_split(split):
        texts, labels = [], []
        for folder, label in [('neg', 0), ('pos', 1)]:
            for path in sorted((DATA_DIR / split / folder).glob('*.txt')):
                texts.append(path.read_text(encoding='utf-8', errors='replace').strip())
                labels.append(label)
        return texts, np.asarray(labels, dtype=np.int32)
    all_train_texts, all_train_labels = read_split('train')
    test_texts, y_test = read_split('test')

train_texts, val_texts, y_train, y_val = train_test_split(
    all_train_texts, all_train_labels, test_size=0.1, random_state=SEED, stratify=all_train_labels
)
print(f'Train: {len(train_texts):,} | Validation: {len(val_texts):,} | Test: {len(test_texts):,}')
print('Train labels [negative, positive]:', np.bincount(y_train))


Đang tải ACL IMDb từ Stanford (~84 MB)...
Không tải được Stanford: URLError(ConnectionRefusedError(111, 'Connection refused'))
Đang thử bản IMDb qua Keras/TensorFlow...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train: 22,500 | Validation: 2,500 | Test: 25,000
Train labels [negative, positive]: [11250 11250]


## 2. Tokenizer và input tensors

Tokenizer chỉ fit trên train. Padding dùng ID `0`; chuỗi dài hơn `MAX_LEN` được cắt ở cuối. Câu ví dụ dùng lại chính tokenizer và cùng cách padding/truncation như dữ liệu huấn luyện.


In [ ]:
MAX_VOCAB = 20000
MAX_LEN = 128
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train_texts)
vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)

def encode_texts(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post', value=0)

X_train = encode_texts(train_texts)
X_val = encode_texts(val_texts)
X_test = encode_texts(test_texts)
print('Vocabulary size:', vocab_size)
print('Input shapes:', X_train.shape, X_val.shape, X_test.shape)


Vocabulary size: 20000
Input shapes: (22500, 128) (2500, 128) (25000, 128)


## 3. Transformer Encoder và classifier

Padding được mask khỏi attention keys và mean pooling. Encoder block được ghép từ Multi-Head Attention, residual connections, LayerNorm và position-wise FFN.


In [ ]:
class SinusoidalPositionEncoding(layers.Layer):
    def call(self, x):
        length, dim = tf.shape(x)[1], tf.shape(x)[2]
        positions = tf.cast(tf.range(length)[:, None], tf.float32)
        indices = tf.cast(tf.range(dim)[None, :], tf.float32)
        rates = tf.pow(10000.0, -(2.0 * tf.floor(indices / 2.0)) / tf.cast(dim, tf.float32))
        angles = positions * rates
        is_even = tf.equal(tf.math.floormod(tf.range(dim), 2), 0)[None, :]
        encoding = tf.where(is_even, tf.sin(angles), tf.cos(angles))[None, :, :]
        return x + tf.cast(encoding, x.dtype)

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=NUM_HEADS, key_dim=EMBED_DIM // NUM_HEADS, dropout=0.1
        )
        self.ffn = models.Sequential([layers.Dense(FF_DIM, activation='relu'), layers.Dense(EMBED_DIM)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(0.1)
        self.dropout2 = layers.Dropout(0.1)
    def call(self, inputs, training=None):
        x, valid_tokens = inputs
        attention_mask = valid_tokens[:, None, :]
        attended = self.attention(query=x, value=x, key=x, attention_mask=attention_mask, training=training)
        x = self.norm1(x + self.dropout1(attended, training=training))
        transformed = self.ffn(x, training=training)
        return self.norm2(x + self.dropout2(transformed, training=training))

class MaskedMeanPooling(layers.Layer):
    def call(self, inputs):
        sequence, valid_tokens = inputs
        mask = tf.cast(valid_tokens[:, :, None], sequence.dtype)
        total = tf.reduce_sum(sequence * mask, axis=1)
        count = tf.maximum(tf.reduce_sum(mask, axis=1), tf.ones_like(tf.reduce_sum(mask, axis=1)))
        return total / count

token_ids = layers.Input(shape=(MAX_LEN,), dtype='int32', name='token_ids')
valid_tokens = layers.Lambda(lambda ids: tf.not_equal(ids, 0), name='valid_tokens')(token_ids)
x = layers.Embedding(vocab_size, EMBED_DIM, name='embedding')(token_ids)
x = SinusoidalPositionEncoding(name='positional_encoding')(x)
x = TransformerEncoderBlock(name='transformer_encoder')([x, valid_tokens])
x = MaskedMeanPooling(name='mean_pooling')([x, valid_tokens])
x = layers.Dropout(0.1)(x)
probability = layers.Dense(1, activation='sigmoid', name='positive_probability')(x)
model = models.Model(token_ids, probability, name='IMDb_Transformer_Classifier')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "IMDb_Transformer_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 128, 64)   │  1,280,000 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 128, 64)   │          0 │ embedding[0][0]   │
│ (SinusoidalPositio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ valid_tokens        │ (None, 128)       │          0 │ token_ids[0][0]   │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 128, 64)   │     33,472 │ positional_encod… │
│ (TransformerEncode… │                   │            │ valid_tokens[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mean_pooling        │ (None, 64)        │          0 │ transformer_enco… │
│ (MaskedMeanPooling) │                   │            │ valid_tokens[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ mean_pooling[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positive_probabili… │ (None, 1)         │         65 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,313,537 (5.01 MB)

 Trainable params: 1,313,537 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Huấn luyện và đánh giá test set


In [ ]:
start_time = time.perf_counter()
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True, verbose=2
)
training_seconds = time.perf_counter() - start_time

test_probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype(np.int32)
print(f'Training time: {training_seconds:.1f} seconds')
print('Accuracy:', round(accuracy_score(y_test, test_predictions), 4))
print('Precision:', round(precision_score(y_test, test_predictions, zero_division=0), 4))
print('Recall:', round(recall_score(y_test, test_predictions, zero_division=0), 4))
print('F1:', round(f1_score(y_test, test_predictions, zero_division=0), 4))
print('Confusion matrix [negative, positive]:\n', confusion_matrix(y_test, test_predictions))
print(classification_report(y_test, test_predictions, target_names=['negative', 'positive'], zero_division=0))


Epoch 1/3
352/352 - 65s - 183ms/step - accuracy: 0.7144 - loss: 0.5233 - val_accuracy: 0.8444 - val_loss: 0.3545
Epoch 2/3
352/352 - 80s - 227ms/step - accuracy: 0.8802 - loss: 0.2884 - val_accuracy: 0.8464 - val_loss: 0.3673
Epoch 3/3
352/352 - 82s - 233ms/step - accuracy: 0.9246 - loss: 0.1912 - val_accuracy: 0.8404 - val_loss: 0.4627
Training time: 248.5 seconds
Accuracy: 0.8071
Precision: 0.842
Recall: 0.7562
F1: 0.7968
Confusion matrix [negative, positive]:
 [[10726  1774]
 [ 3048  9452]]
              precision    recall  f1-score   support

    negative       0.78      0.86      0.82     12500
    positive       0.84      0.76      0.80     12500

    accuracy                           0.81     25000
   macro avg       0.81      0.81      0.81     25000
weighted avg       0.81      0.81      0.81     25000



## 5. Thử dự đoán văn bản mới

Các mẫu bao gồm câu ngắn, câu dài, phủ định, tương phản và cảm xúc mơ hồ. Kiểm tra xác suất cùng dự đoán của mô hình; các câu này không phải bằng chứng cho hiệu năng tổng quát trên từng hiện tượng.


In [ ]:
examples = [
    {'Case': 'Câu ngắn', 'Text': 'A wonderful film.'},
    {'Case': 'Câu dài', 'Text': 'The movie begins with a fascinating premise and several memorable performances. The cinematography is beautiful, and a few scenes are genuinely moving. However, the middle section repeats the same ideas, some characters are underdeveloped, and the ending feels rushed. I enjoyed parts of it, but the whole experience is uneven.'},
    {'Case': 'Phủ định', 'Text': 'I did not enjoy this movie; the story was dull and the acting felt lifeless.'},
    {'Case': 'Tương phản', 'Text': 'The first half was slow and confusing, but the final act was so moving that I left the theater impressed.'},
    {'Case': 'Cảm xúc mơ hồ', 'Text': 'It was fine. Not especially good, not especially bad, and I probably will not think about it again.'},
]

example_texts = [item['Text'] for item in examples]
example_sequences = tokenizer.texts_to_sequences(example_texts)
example_ids = pad_sequences(example_sequences, maxlen=MAX_LEN, padding='post', truncating='post', value=0)
example_probabilities = model.predict(example_ids, verbose=0).ravel()

prediction_rows = []
for item, sequence, positive_prob in zip(examples, example_sequences, example_probabilities):
    predicted_label = 'positive' if positive_prob >= 0.5 else 'negative'
    prediction_rows.append({
        'Case': item['Case'],
        'Tokens retained': min(len(sequence), MAX_LEN),
        'Prediction': predicted_label,
        'P(positive)': float(positive_prob),
        'Text': item['Text'],
    })

predictions_df = pd.DataFrame(prediction_rows)
display(predictions_df.round({'P(positive)': 4}))


,Case,Tokens retained,Prediction,P(positive),Text
0,Câu ngắn,3,positive,0.9998,A wonderful film.
1,Câu dài,50,positive,0.9992,The movie begins with a fascinating premise an...
2,Phủ định,15,negative,0.1908,I did not enjoy this movie; the story was dull...
3,Tương phản,20,negative,0.0008,"The first half was slow and confusing, but the..."
4,Cảm xúc mơ hồ,18,positive,0.9426,"It was fine. Not especially good, not especial..."
